# Silver — ANP Automotivos

Este notebook executa e valida o processamento da camada Silver dos dados de preços de combustíveis automotivos da ANP.

A camada Silver realiza a limpeza, padronização, tipagem e validação dos dados da Bronze, persistindo as partições válidas em formato Parquet.

Também é validada a integração geográfica com a dimensão oficial de municípios do IBGE, utilizada posteriormente na construção da camada Gold.

In [7]:
from pathlib import Path

import polars as pl

from insightfuel_data_platform.ingestion.ibge import (
    baixar_municipios_ibge,
)
from insightfuel_data_platform.pipelines.anp import (
    processar_particoes_anp,
)
from insightfuel_data_platform.transformation.anp import (
    enriquecer_com_codigo_ibge,
)
from insightfuel_data_platform.transformation.ibge import (
    transformar_municipios_ibge,
)

## 1. Processamento Bronze → Silver

As partições disponíveis na Bronze são descobertas automaticamente.

Partições que já possuem saída na Silver não são reprocessadas, permitindo uma carga incremental básica.

In [8]:
pasta_bronze = Path("../data/bronze/anp/automotivos")
pasta_silver = Path("../data/silver/anp/automotivos")

In [9]:
processados = processar_particoes_anp(
    pasta_bronze,
    pasta_silver,
)

if processados:
    print("Partições processadas:")

    for caminho in processados:
        print(caminho)
else:
    print("Nenhuma nova partição para processar.")

Nenhuma nova partição para processar.


## 2. Verificação das partições Silver

Confere as partições Parquet atualmente publicadas na camada Silver.

In [10]:
arquivos_silver = sorted(
    pasta_silver.rglob("*.parquet")
)

for arquivo in arquivos_silver:
    df_silver = pl.read_parquet(arquivo)

    print(
        arquivo,
        "| linhas:", df_silver.height,
        "| colunas:", df_silver.width,
    )

../data/silver/anp/automotivos/ano=2023/semestre=1/dados.parquet | linhas: 431576 | colunas: 11
../data/silver/anp/automotivos/ano=2023/semestre=2/dados.parquet | linhas: 472424 | colunas: 11
../data/silver/anp/automotivos/ano=2024/semestre=1/dados.parquet | linhas: 477144 | colunas: 11
../data/silver/anp/automotivos/ano=2024/semestre=2/dados.parquet | linhas: 421382 | colunas: 11
../data/silver/anp/automotivos/ano=2025/semestre=1/dados.parquet | linhas: 420409 | colunas: 11
../data/silver/anp/automotivos/ano=2025/semestre=2/dados.parquet | linhas: 384208 | colunas: 11


## 3. Dimensão de municípios do IBGE

Os municípios são obtidos da API de Localidades do IBGE e transformados em uma dimensão geográfica contendo código IBGE, município, UF e região.

Uma chave textual normalizada é utilizada internamente apenas para associar os municípios da ANP aos municípios oficiais do IBGE.

In [11]:
df_ibge = baixar_municipios_ibge()

df_municipios = transformar_municipios_ibge(
    df_ibge
)

print(df_municipios.shape)
print(df_municipios.null_count())

df_municipios.head()

(5571, 5)
shape: (1, 5)
┌─────────────┬───────────┬─────┬────────┬─────────────────┐
│ codigo_ibge ┆ municipio ┆ uf  ┆ regiao ┆ municipio_match │
│ ---         ┆ ---       ┆ --- ┆ ---    ┆ ---             │
│ u32         ┆ u32       ┆ u32 ┆ u32    ┆ u32             │
╞═════════════╪═══════════╪═════╪════════╪═════════════════╡
│ 0           ┆ 0         ┆ 0   ┆ 0      ┆ 0               │
└─────────────┴───────────┴─────┴────────┴─────────────────┘


codigo_ibge,municipio,uf,regiao,municipio_match
str,str,str,str,str
"""1100015""","""Alta Floresta D'Oeste""","""RO""","""Norte""","""ALTA FLORESTA DOESTE"""
"""1100023""","""Ariquemes""","""RO""","""Norte""","""ARIQUEMES"""
"""1100031""","""Cabixi""","""RO""","""Norte""","""CABIXI"""
"""1100049""","""Cacoal""","""RO""","""Norte""","""CACOAL"""
"""1100056""","""Cerejeiras""","""RO""","""Norte""","""CEREJEIRAS"""


## 4. Validação da integração ANP × IBGE

Cada partição Silver da ANP é associada à dimensão de municípios por
UF e município normalizado.

O objetivo desta validação é garantir que todas as observações da ANP
possam receber um código oficial de município do IBGE antes da construção
da camada Gold.

In [12]:
for arquivo in arquivos_silver:
    df_silver = pl.read_parquet(arquivo)

    df_enriquecido = enriquecer_com_codigo_ibge(
        df_silver,
        df_municipios,
    )

    sem_codigo_ibge = (
        df_enriquecido
        .filter(pl.col("codigo_ibge").is_null())
        .height
    )

    print(
        arquivo,
        "| linhas:", df_enriquecido.height,
        "| sem codigo_ibge:", sem_codigo_ibge,
    )

../data/silver/anp/automotivos/ano=2023/semestre=1/dados.parquet | linhas: 431576 | sem codigo_ibge: 0
../data/silver/anp/automotivos/ano=2023/semestre=2/dados.parquet | linhas: 472424 | sem codigo_ibge: 0
../data/silver/anp/automotivos/ano=2024/semestre=1/dados.parquet | linhas: 477144 | sem codigo_ibge: 0
../data/silver/anp/automotivos/ano=2024/semestre=2/dados.parquet | linhas: 421382 | sem codigo_ibge: 0
../data/silver/anp/automotivos/ano=2025/semestre=1/dados.parquet | linhas: 420409 | sem codigo_ibge: 0
../data/silver/anp/automotivos/ano=2025/semestre=2/dados.parquet | linhas: 384208 | sem codigo_ibge: 0


In [13]:
df_enriquecido.select(
    "codigo_ibge",
    "municipio",
    "uf",
    "regiao",
    "produto",
    "data_coleta",
    "valor_venda",
).head(10)

codigo_ibge,municipio,uf,regiao,produto,data_coleta,valor_venda
str,str,str,str,str,date,"decimal[10,2]"
"""3111200""","""CAMPO BELO""","""MG""","""SE""","""DIESEL S10""",2025-12-08,6.29
"""1501402""","""BELEM""","""PA""","""N""","""GASOLINA""",2025-09-08,6.29
"""3534401""","""OSASCO""","""SP""","""SE""","""ETANOL""",2025-11-10,3.99
"""2903201""","""BARREIRAS""","""BA""","""NE""","""GASOLINA ADITIVADA""",2025-08-06,6.49
"""3529401""","""MAUA""","""SP""","""SE""","""GASOLINA ADITIVADA""",2025-12-15,5.99
"""5212501""","""LUZIANIA""","""GO""","""CO""","""DIESEL S10""",2025-10-02,6.29
"""3529005""","""MARILIA""","""SP""","""SE""","""ETANOL""",2025-10-24,3.99
"""3523909""","""ITU""","""SP""","""SE""","""GASOLINA ADITIVADA""",2025-08-06,6.47
"""3201308""","""CARIACICA""","""ES""","""SE""","""ETANOL""",2025-11-18,4.49
